# Use Case Description

Automate key phases of incident investigation using a coordinated set of LLM-driven agents. These agents handle tasks such as triaging, summarizing, planning next steps, and generating recommendations, ultimately producing detailed investigation reports for SOC teams with minimal manual effort.

## Model used for this use case
In this implementation, we utilized the Reasoning Model. <br>

## SetUp

We provide four different setup methods in this example. 

1. [Quickstart with Transformers](https://github.com/cisco-foundation-ai/cookbook/blob/main/1_quickstarts/Preview_Quickstart_reasoning_model.ipynb)
2. [Deploy on Sagemaker](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb)
3. [Deploy on Baseten](https://github.com/cisco-foundation-ai/cookbook/tree/main/3_adoptions/deployment/baseten)
4. [Deploy on Bedrock](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/deploy.ipynb)

Make sure the helper file [inference_clients.py](https://github.com/cisco-foundation-ai/cookbook/blob/main/2_examples/inference_clients.py) is located in the same directory of this notebook and then: <br>
1. **uncomment** the preferred setup method in the cell below  <br> 
2. **Fill in** the necessary arguments based on your deployment type and run the following cell to initiate the helper.


In [ ]:
from inference_clients import ReasoningModelClient
from IPython.display import display, Markdown

## Uncomment for Transformers Deployment
# client = ReasoningModelClient.from_transformers(
#     model_id="fdtn-ai/Foundation-Sec-8B-Reasoning"  # Hugging Face model name. For reasoning model: "fdtn-ai/Foundation-Sec-8B-Reasoning",
# )

## Uncomment for Sagemaker Deployment
# client = ReasoningModelClient.from_sagemaker(
#     endpoint_name=''  # Fill in your sagemaker deployment endpoint if applicable
# )

## Uncomment for Baseten Deployment
# client = ReasoningModelClient.from_baseten(
#     endpoint_url='',  # Fill in your baseten deployment endpoint if applicable. Example: https://XXXXX.api.baseten.co/environments/production/sync/v1/chat/completions,
#     api_key=''  # Fill in your baseten API key if applicable
# )

## Uncomment for Bedrock Deployment
# client = ReasoningModelClient.from_bedrock(
#     region='',
#     model_arn=''  # copy from deploy notebook or: aws bedrock list-imported-models
# )

## Step 1: Summarize Incident

In [3]:
def make_prompt_summarize_incident(metadata: dict, alert_logs: str) -> str:

    return (
        "You are a senior SOC analyst assisting with incident triage. "
        "Your task is to read the incident metadata and alert logs, and provide a clear summary of what occurred.\n\n"
        "Instructions:\n"
        "- Highlight the sequence of events (inferred from timestamps).\n"
        "- Think deeply about cause and effect and how artifacts relate to one another.\n"
        "- Mention key attack techniques used (if inferable from logs).\n"
        "- Describe how the attack began and progressed.\n"
        "- Use clear and concise language appropriate for L1/L2 analysts.\n\n"
        f"Incident Metadata:\n{metadata}\n\n"
        f"Alert Logs:\n{alert_logs}\n\n"
        "Summarize what happened in this incident in a few sentences"
    )


In [4]:
incident_meta = {
    "incident_id": "INC-1024",
    "type": "Unauthorized Access",
    "severity": "High",
    "timestamp": "2025-04-09T10:30:00Z"
}
raw_logs = """2025-04-09 10:00:23 - Alert: 5 failed login attempts for user 'alice' on host 'WS123'
2025-04-09 10:05:10 - Alert: Suspicious PowerShell execution on 'WS123' by 'alice' (malicious script blocked)
2025-04-09 10:10:45 - Alert: Process dumping LSASS memory on 'WS123' (possible credential theft)
2025-04-09 10:15:00 - Alert: Successful login of 'alice' to server 'DC1' from host 'WS123'"""

In [ ]:
reasoning, generated_summary = client.chat_generation(make_prompt_summarize_incident(incident_meta, raw_logs))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(generated_summary))

=== Reasoning (chain-of-thought) ===



Alright, let me tackle this incident summary request step by step. The user wants a clear breakdown of an unauthorized access incident based on provided metadata and alerts. 

First, I need to parse the incident metadata: high severity unauthorized access starting at 10:30 AM UTC. That sets the timeline context. Now examining the alert logs chronologically:

The first alert shows five failed login attempts against Alice on workstation WS123 at 10:00 AM. This suggests brute-force attacks targeting her account. Then at 10:05 AM, suspicious PowerShell activity was detected - likely attacker testing system capabilities. 

At 10:10 AM, LSASS process dumping occurred - classic credential harvesting technique. Finally, successful login to DC1 from WS123 at 10:15 AM confirms initial access achieved. 

Connecting these dots: The attack progression clearly follows MITRE ATT&CK T1110 (Brute Force) → T1059.001 (PowerShell) → T1003.001 (LSASS Dumping) → T1078 (Valid Accounts). The lateral movement to domain controller DC1 indicates privilege escalation goals.

Key takeaway: Attackers compromised Alice's credentials through brute force, then escalated privileges via credential dumping before moving laterally to critical infrastructure. The timeline shows rapid progression within just 55 minutes, emphasizing urgency. 

I'll structure the response to mirror this chronological flow while highlighting the attack chain and critical artifacts. Should keep it concise but include enough technical detail for actionable insights.


<mark>=== Answer ===</mark>



Based on the provided incident metadata and alert logs, here's a clear summary of the event:

**Summary:**  
On April 9th at approximately 10:30 AM UTC, a high-severity unauthorized access incident occurred involving user "alice" and host "WS123". The attack followed a clear progression:  

1. **Initial Compromise Attempt**: At 10:00 AM, multiple failed login attempts targeted "alice" on "WS123", indicating brute-force attacks.  
2. **Execution of Malicious Activity**: Shortly after (10:05 AM), suspicious PowerShell commands were executed by "alice" on "WS123", suggesting post-compromise reconnaissance or tool deployment.  
3. **Credential Theft**: At 10:10 AM, the LSASS process was dumped on "WS123" – a tactic to extract credentials stored in memory.  
4. **Lateral Movement & Success**: By 10:15 AM, "alice" successfully logged into the Domain Controller ("DC1") from "WS123", confirming initial access was gained and lateral movement initiated.  

**Attack Chain Inference**:  
The sequence aligns with MITRE ATT&CK techniques:  
- **T1110 (Brute Force)**: Failed logins for "alice".  
- **T1059.001 (Command and Scripting Interpreter: PowerShell)**: Suspicious script execution.  
- **T1003.001 (OS Credential Dumping: LSASS Memory)**: Credential theft attempt.  
- **T1078 (Valid Accounts)**: Successful login to "DC1" using stolen/compromised credentials.  

This indicates attackers successfully breached "alice"'s account, stole credentials, and escalated privileges to compromise critical infrastructure (Domain Controller).

##  Step 2: Identify Impacted Assets, Users, and MITRE Tactics

Identify impacted assets, users, and MITRE ATT&CK tactics/techniques from incident data. 

In [ ]:
prompt = (
    "You are a cybersecurity expert specializing in threat analysis and incident response. Analyze the following incident logs and metadata, then:\n"
    "- List impacted host systems and IPs\n"
    "- List impacted user accounts\n"
    "- Identify MITRE ATT&CK tactics and techniques observed (with names or IDs)\n\n"
    f"Metadata: {incident_meta}\n"
    f"Logs:\n{raw_logs}\n\n"
    "Provide the result as a JSON object with no comments where"
    "- keys are 'impacted_hosts', 'impacted_users', 'tactics', all of which are lists"
    "- each tactic in 'tactics' should have 'name' and 'techniques' keys"
    "- each technique in 'techniques' should have 'id' and 'name' keys"
    "Only output JSON"
)

reasoning, generated_entities = client.chat_generation(prompt)

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(generated_entities))

=== Reasoning (chain-of-thought) ===



My process begins by dissecting the provided incident logs and metadata to extract critical information regarding affected assets and attacker actions. The goal is to categorize these observations into specific hosts, users, and MITRE ATT&CK tactics and techniques, ultimately structuring this data into a JSON format as requested.

First, I identify the **impacted host systems**. Reviewing the logs, I see explicit mentions of 'WS123' in multiple entries: failed logins, suspicious PowerShell execution, and LSASS memory dumping. This clearly indicates 'WS123' as an impacted host. No other unique host identifiers appear in the logs, so I'll list just 'WS123'.

Next, I determine the **impacted user accounts**. The logs explicitly name 'alice' in connection with the initial failed login attempts and subsequent suspicious activities originating from 'WS123'. Therefore, 'alice' is the sole impacted user identified in the logs.

Moving on to **MITRE ATT&CK tactics and techniques**, I analyze each alert chronologically:

1.  **"Alert: 5 failed login attempts for user 'alice' on host 'WS123'"**: This event signifies repeated, unsuccessful authentication attempts against a legitimate account. This behavior aligns with the **TA0006 Credential Access** tactic, specifically the **T1110 Brute Force** technique, as the adversary is attempting to guess credentials through repeated trials.
2.  **"Alert: Suspicious PowerShell execution on 'WS123' by 'alice' (malicious script blocked)"**: The detection of malicious PowerShell activity points towards the use of scripting capabilities within the operating system. This falls under the **TA0002 Execution** tactic, with the **T1059.001 Command and Scripting Interpreter: PowerShell** technique being the most fitting description for using PowerShell for malicious purposes.
3.  **"Alert: Process dumping LSASS memory on 'WS123' (possible credential theft)"**: The act of dumping LSASS memory is a known method for adversaries to extract credentials stored in the Local Security Authority Subsystem Service. This action directly maps to the **TA0006 Credential Access** tactic, utilizing the **T1003.001 OS Credential Dumping: LSASS Memory** technique.

Finally, I compile these findings into the specified JSON structure. The 'impacted_hosts' array contains ['WS123'], 'impacted_users' contains ['alice'], and the 'tactics' array includes objects for TA0006 (Credential Access) and TA0002 (Execution), each detailing their respective techniques with their IDs and names. This structured output provides a clear and concise summary of the incident's key elements.


<mark>=== Answer ===</mark>



{
  "impacted_hosts": ["WS123"],
  "impacted_users": ["alice"],
  "tactics": [
    {
      "name": "Credential Access",
      "techniques": [
        {"id": "T1110", "name": "Brute Force"},
        {"id": "T1003.001", "name": "OS Credential Dumping: LSASS Memory"}
      ]
    },
    {
      "name": "Execution",
      "techniques": [
        {"id": "T1059.001", "name": "Command and Scripting Interpreter: PowerShell"}
      ]
    }
  ]
}
</details>

Cleanse the response data and read as json file

In [7]:
import json
import re

match = re.search(r'\{[\s\S]*\}', generated_entities)
if match:
    try:
        generated_entities = json.loads(match.group(0))
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
else:
    print("No JSON object found in response.")
print(json.dumps(generated_entities, indent=2))

{
  "impacted_hosts": [
    "WS123"
  ],
  "impacted_users": [
    "alice"
  ],
  "tactics": [
    {
      "name": "Credential Access",
      "techniques": [
        {
          "id": "T1110",
          "name": "Brute Force"
        },
        {
          "id": "T1003.001",
          "name": "OS Credential Dumping: LSASS Memory"
        }
      ]
    },
    {
      "name": "Execution",
      "techniques": [
        {
          "id": "T1059.001",
          "name": "Command and Scripting Interpreter: PowerShell"
        }
      ]
    }
  ]
}


 ## Step 3: Recommend Remediation Steps and Next Investigative Actions

 Recommend remediation steps and next investigative actions based on the incident summary and entities.

In [8]:
def make_prompt_for_recommend_actions(summary, entities) -> dict:
    impacted_hosts = ", ".join(entities.get("impacted_hosts", []))
    impacted_users = ", ".join(entities.get("impacted_users", []))
    tactics_list = [
        f"{tactic['name']}: {technique['name']} ({technique['id']})"
        for tactic in entities.get("tactics", [])
        for technique in tactic.get("techniques", [])
    ]

    context = (
        f"Incident Summary: {summary}\n"
        f"Impacted Hosts: {impacted_hosts}\n"
        f"Impacted Users: {impacted_users}\n"
        f"Observed Tactics: {tactics_list}\n\n"
    )

    return (
        "You are a SOC incident response assistant. Based on the incident details, respond strictly in the following JSON format:\n"
        '{\n  "remediation_steps": ["..."],\n  "next_steps": ["..."]\n}\n\n'
        + context
    )

In [ ]:
reasoning, _generated_recommend_actions = client.chat_generation(make_prompt_for_recommend_actions(generated_summary, generated_entities))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(_generated_recommend_actions))

=== Reasoning (chain-of-thought) ===



Okay, let me approach this incident step-by-step as a cybersecurity analyst would. The user has shared an incident summary with specific timestamps and observed behaviors. My primary goal is to provide actionable remediation steps while maintaining strict adherence to the required JSON output format.

First, I need to analyze the attack chain chronologically: 
- Initial brute force attempts targeting user "alice" on workstation WS123 starting at 10:00 AM
- Followed by suspicious PowerShell activity at 10:05 AM
- Then LSASS dumping at 10:10 AM - this is critical as it shows credential extraction
- Finally successful DC login at 10:15 AM indicating privilege escalation

The key indicators here are:
1. User "alice" being compromised (likely via password spraying or weak password)
2. Credential theft via LSASS dumping (highly sensitive technique)
3. Lateral movement to Domain Controller showing domain-wide impact

For remediation, I should focus on:
- Immediate containment of affected systems
- Credential rotation across all accounts
- Forensic preservation of evidence
- Prevention measures against similar attacks

Next steps should include:
- Detailed forensic investigation
- Vulnerability assessment
- Security policy review
- User awareness training

I'll structure my response in the exact JSON format requested, ensuring each remediation step addresses a specific phase of the attack. The next steps will cover both immediate actions and longer-term improvements.


<mark>=== Answer ===</mark>



{
  "remediation_steps": [
    "Isolate WS123 immediately from the network to prevent further lateral movement.",
    "Reset passwords for all users, especially privileged accounts, and enforce multi-factor authentication (MFA) enforcement.",
    "Conduct forensic imaging of WS123 and DC1 to preserve evidence of malicious activity.",
    "Disable RDP and WMI services on WS123 and DC1 temporarily until patched.",
    "Audit all domain accounts for suspicious activity and reset any potentially compromised credentials."
  ],
  "next_steps": [
    "Perform detailed forensic analysis of LSASS dumps to identify stolen credentials.",
    "Review firewall rules and segment internal networks to limit lateral movement paths.",
    "Patch all endpoints and servers against known vulnerabilities exploited in this campaign.",
    "Implement enhanced monitoring for PowerShell execution and credential dumping activities.",
    "Provide mandatory security awareness training focusing on phishing and password hygiene."
  ]
}

Cleanse the response data and read as json file

In [10]:
try:
    generated_recommend_actions = json.loads(_generated_recommend_actions)
    print(generated_recommend_actions)
except:
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", _generated_recommend_actions, re.DOTALL)
    if match:
        generated_recommend_actions = json.loads(match.group(1))
        print(json.dumps(generated_recommend_actions, indent=2))
    else:
        print("JSON was not correctly generated")

{'remediation_steps': ['Isolate WS123 immediately from the network to prevent further lateral movement.', 'Reset passwords for all users, especially privileged accounts, and enforce multi-factor authentication (MFA) enforcement.', 'Conduct forensic imaging of WS123 and DC1 to preserve evidence of malicious activity.', 'Disable RDP and WMI services on WS123 and DC1 temporarily until patched.', 'Audit all domain accounts for suspicious activity and reset any potentially compromised credentials.'], 'next_steps': ['Perform detailed forensic analysis of LSASS dumps to identify stolen credentials.', 'Review firewall rules and segment internal networks to limit lateral movement paths.', 'Patch all endpoints and servers against known vulnerabilities exploited in this campaign.', 'Implement enhanced monitoring for PowerShell execution and credential dumping activities.', 'Provide mandatory security awareness training focusing on phishing and password hygiene.']}


## Step 4: Produce a Structured Incident Investigation Report
Compile a structured incident report using the summary, entities, and recommended actions.
Returns a formatted report as a text (could be markdown or plain text).

In [11]:
def make_report_prompt(metadata: dict, summary: str, entities: dict, actions: dict) -> str:
    """
    Compile a structured incident report using the summary, entities, and recommended actions.
    Returns a formatted report as a text (could be markdown or plain text).
    """
    hosts = ", ".join(entities.get("impacted_hosts", []))
    users = ", ".join(entities.get("impacted_users", []))
    tactics = [
        f"{tactic['name']}: {technique['name']} ({technique['id']})"
        for tactic in entities.get("tactics", [])
        for technique in tactic.get("techniques", [])
    ]
    remediation_list = "- " + "\n- ".join(
        step["text"] if isinstance(step, dict) and "text" in step else str(step)
        for step in actions.get("remediation_steps", [])
    )
    next_steps_list = "- " + "\n- ".join(
        step["text"] if isinstance(step, dict) and "text" in step else str(step)
        for step in actions.get("next_steps", [])
    )

    return (
        "You are a SOC analyst assistant tasked with writing the final incident report. "
        "Use the information below to fill out each section of the report.\n\n"
        f"Incident Metadata: {metadata}\n"
        f"Incident Summary: {summary}\n"
        f"Impacted Hosts: {hosts}\n"
        f"Impacted Users: {users}\n"
        f"Observed Tactics/Techniques: {tactics}\n"
        f"Remediation Steps Taken:\n{remediation_list}\n"
        f"Next Investigation Steps:\n{next_steps_list}\n\n"
        "Now format the incident report with the following sections:\n"
        "## Incident Summary\n"
        "*(A brief overview of the incident.)*\n"
        "## Impacted Assets and Users\n"
        "*(Which systems and accounts were affected.)*\n"
        "## Adversary Tactics and Techniques\n"
        "*(ATT&CK tactics/techniques observed in this incident.)*\n"
        "### Incident Details (Timeline)\n"
        "*(Chronological log of key events.)*\n\n"
        "## Remediation Actions Implemented\n"
        "*(How we contained/mitigated the incident.)*\n"
        "## Next Steps and Recommendations\n"
        "*(Further investigation steps or preventive measures.)*\n"
        "Draft the report now, incorporating all provided details in the appropriate sections."
        "Return in Markdown format"
    )


In [ ]:
reasoning, report = client.chat_generation(make_report_prompt(incident_meta, generated_summary, generated_entities, generated_recommend_actions))
print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(report))

=== Reasoning (chain-of-thought) ===



Okay, I need to draft an incident report based on the provided incident metadata and details. Let me break down the requirements first.

The user wants a structured report with specific sections: Incident Summary, Impacted Assets and Users, Adversary Tactics and Techniques, Incident Details (Timeline), Remediation Actions Implemented, and Next Steps and Recommendations. All content needs to come from the given incident data.

Looking at the incident metadata: 
- Incident ID: INC-1024
- Type: Unauthorized Access
- Severity: High
- Timestamp: 2025-04-09T10:30:00Z

The incident summary already provides a good starting point: 
- Occurred on April 9th at 10:30 AM UTC
- Involved user "alice" and host "WS123"
- Followed a clear progression: brute force → PowerShell execution → LSASS dump → lateral movement to DC1

For the impacted assets section, I'll list WS123 and DC1 clearly. User "alice" should be mentioned as both victim and attacker vector.

Adversary tactics section needs to map to MITRE ATT&CK: T1110 (brute force), T1059.001 (PowerShell), T1003.001 (LSASS dump), and T1078 (valid accounts). The timeline should show exact timestamps: 10:00AM (failed logins), 10:05AM (PowerShell), 10:10AM (LSASS dump), 10:15AM (lateral move).

Remediation actions should cover isolation, password resets, forensics, disabling services, auditing accounts, and patching. Next steps include forensic analysis, network segmentation, patching, monitoring enhancements, and training.

I need to ensure all dates/times use ISO 8601 format consistently. The language should remain professional yet accessible. Since this is a simulated report, I won't add fictional details beyond what's provided. The structure must follow the requested markdown headings precisely.


<mark>=== Answer ===</mark>



Here is the formatted incident report based on the provided details:

---

# Incident Report: Unauthorized Access - INC-1024

## Incident Summary
On April 9th, 2025, at approximately 10:30 AM UTC, a high-severity unauthorized access incident occurred targeting user "alice" and host "WS123". Attackers leveraged a combination of brute-force techniques, credential dumping, and lateral movement to gain control over critical infrastructure. Initial compromise involved brute-force attacks against "alice"'s account, followed by successful credential theft via LSASS memory dumping, culminating in privilege escalation through lateral movement to the Domain Controller ("DC1").

## Impacted Assets and Users
- **Hosts Affected**: WS123 (Workstation), DC1 (Domain Controller)  
- **Users Affected**: Alice (account compromised; used as initial foothold)  

## Adversary Tactics and Techniques
Attackers employed the following MITRE ATT&CK techniques:  
- **Credential Access**:  
  - **T1110 (Brute Force)**: Multiple failed login attempts targeting "alice" on WS123.  
  - **T1003.001 (OS Credential Dumping: LSASS Memory)**: Dumping of LSASS process memory on WS123 to extract credentials.  
- **Execution**:  
  - **T1059.001 (Command and Scripting Interpreter: PowerShell)**: Execution of suspicious PowerShell commands on WS123.  
- **Persistence/Lateral Movement**:  
  - **T1078 (Valid Accounts)**: Successful login to DC1 using compromised credentials from WS123.  

### Incident Details (Timeline)
| Time                  | Event                                                                 |
|-----------------------|-----------------------------------------------------------------------|
| 2025-04-09T10:00:00Z | First failed login attempts targeting "alice" on WS123 (brute force). |
| 2025-04-09T10:05:00Z | Suspicious PowerShell commands executed on WS123.                    |
| 2025-04-09T10:10:00Z | LSASS process dumped on WS123 (credential harvesting).                |
| 2025-04-09T10:15:00Z | "alice" logged into DC1 from WS123 (privilege escalation).            |

## Remediation Actions Implemented
Immediate containment and mitigation steps included:  
1. **Isolation**: WS123 disconnected from the network to halt lateral movement.  
2. **Account Security**: Password resets enforced for all users, especially privileged accounts; MFA enabled across the environment.  
3. **Forensic Preservation**: Full disk images created for WS123 and DC1 to capture evidence.  
4. **Service Disruption**: RDP and WMI disabled on WS123 and DC1 pending patching.  
5. **Audit**: Active Directory audit logs reviewed; all domain accounts checked for anomalies.  

## Next Steps and Recommendations
To prevent recurrence and address root causes:  
1. **Forensic Analysis**: Investigate LSASS dumps to identify stolen credentials and scope impact.  
2. **Network Hardening**: Segment internal networks to restrict lateral movement paths; review firewall rules.  
3. **Vulnerability Management**: Patch all endpoints/servers against known exploits (e.g., CVE-2023-XXXXX if applicable).  
4. **Enhanced Monitoring**: Deploy SIEM alerts for anomalous PowerShell activity and credential dumping.  
5. **Security Training**: Mandatory phishing simulations and password hygiene workshops for employees.  

---